# YOLOv8 Bone Fracture Detection - Model Training
Modified to run on Colab

This notebook trains a YOLOv8 model on the cleaned bone fracture detection dataset.

In [1]:
# Install ultralytics (YOLOv8)
%pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.1 MB/s eta 0:00:00


In [2]:
import os
from ultralytics import YOLO
import matplotlib.pyplot as plt
import yaml
import torch

# Check what devices are available
print("Device Check:")
print(f"  CUDA available: {torch.cuda.is_available()}")
print(f"  MPS (Apple GPU) available: {torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else False}")
print(f"  CPU: Always available")

# Determine best device
if torch.cuda.is_available():
    device = 0
    device_name = "CUDA GPU"
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'  # Apple Silicon GPU
    device_name = "Apple MPS (GPU)"
else:
    device = 'cpu'
    device_name = "CPU"

print(f"\nUsing device: {device_name} ({device})")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Device Check:
  CUDA available: True
  MPS (Apple GPU) available: False
  CPU: Always available

Using device: CUDA GPU (0)


Retriving the data

In [3]:
import kagglehub
import os
import yaml
from pathlib import Path

# Download dataset
top_level_path = kagglehub.dataset_download(
    "pkdarabi/bone-fracture-detection-computer-vision-project",
    force_download=True
)

print("Top level path:", top_level_path)
print("Contents:", os.listdir(top_level_path))

# Identify dataset folder automatically
dataset_folder = os.listdir(top_level_path)[0]
data_path = os.path.join(top_level_path, dataset_folder)

print("Dataset path:", data_path)
print("Dataset contents:", os.listdir(data_path))

# Convert to Path object
dataset_dir = Path(data_path)

# Locate YOLO dataset config
data_yaml_path = dataset_dir / "data.yaml"

# Verify file exists
if not data_yaml_path.exists():
    raise FileNotFoundError(f"Dataset YAML not found: {data_yaml_path}")

# Load dataset configuration
with open(data_yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

print("\nDataset Configuration:")
print(f"  Number of classes: {data_config['nc']}")
print(f"  Class names: {data_config['names']}")
print(f"  Train path: {data_config['train']}")
print(f"  Val path: {data_config['val']}")
print(f"  Test path: {data_config['test']}")

100%|██████████| 84.1M/84.1M [00:00<00:00, 144MB/s]

Extracting files...


Top level path: /root/.cache/kagglehub/datasets/pkdarabi/bone-fracture-detection-computer-vision-project/versions/2
Contents: ['BoneFractureYolo8', 'bone fracture detection.v4-v4.yolov8']
Dataset path: /root/.cache/kagglehub/datasets/pkdarabi/bone-fracture-detection-computer-vision-project/versions/2/BoneFractureYolo8
Dataset contents: ['test', 'train', 'README.dataset.txt', 'valid', 'data.yaml']

Dataset Configuration:
  Number of classes: 7
  Class names: ['elbow positive', 'fingers positive', 'forearm fracture', 'humerus fracture', 'humerus', 'shoulder fracture', 'wrist positive']
  Train path: ../train/images
  Val path: ../valid/images
  Test path: ../test/images


Choose the model version to run

In [4]:
# Initialize YOLOv8 model
# Options: 'yolov8n.pt' (nano), 'yolov8s.pt' (small), 'yolov8m.pt' (medium),
#          'yolov8l.pt' (large), 'yolov8x.pt' (xlarge)
# Start with 'yolov8n.pt' for faster training, or 'yolov8s.pt' for better accuracy
model = YOLO('yolov8n.pt')  # Change to yolov8s.pt, yolov8m.pt, etc. for better accuracy
print(f"Model initialized: {model.model_name}")

Model initialized: yolov8n.pt


Training the model

In [ ]:
# Train the model
# Device is automatically detected from Cell 2 (will use MPS/GPU if available, otherwise CPU)
results = model.train(
    data=data_yaml_path,
    epochs=100,              # Number of training epochs
    imgsz=640,              # Image size
    batch=16 if device != 'cpu' else 8,  # Larger batch for GPU, smaller for CPU
    name='bone_fracture_yolov8',  # Project name
    project='runs/detect',  # Project directory
    patience=20,            # Early stopping patience
    save=True,              # Save checkpoints
    plots=True,             # Generate training plots
    val=True,               # Validate during training
    device=device,          # Uses device detected in Cell 2 (MPS/GPU/CPU)
    workers=8 if device != 'cpu' else 4,  # More workers for GPU
)

Ultralytics 8.4.26 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/.cache/kagglehub/datasets/pkdarabi/bone-fracture-detection-computer-vision-project/versions/2/BoneFractureYolo8/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=bone_fracture_yolo

## Training Results

After training, the model will be saved in `runs/detect/bone_fracture_yolov8/weights/best.pt`

Troubleshooting

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
!find /content/drive/MyDrive -name "best_yolo_model.pt"

/content/drive/MyDrive/UWaterloo/4A/MSE 446/Project/best_yolo_model.pt
^C


In [24]:
import os

# Specify the directory path ('.' refers to the current directory)
directory_path = '/content/drive/MyDrive/UWaterloo/4A/MSE 446/Project'

# Get the list of all files and folders
entries = os.listdir(directory_path)

# Print each entry on a new line
for entry in entries:
    print(entry)

train_model_jenna.ipynb
.gitignore
README.md
yolov8n.pt
runs
.ipynb_checkpoints
rcnn_train_model.ipynb
rcnn_model.ipynb
best_yolo_model.pt
train_model_v11.ipynb
kagglehub
EDA_jenna.ipynb


In [26]:
# Load the best model from training
best_model_path = '/content/drive/MyDrive/UWaterloo/4A/MSE 446/Project/best_yolo_model.pt'
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    print(f"Loaded best model from: {best_model_path}")
else:
    print(f"Model not found at {best_model_path}. Training may still be in progress.")

Loaded best model from: /content/drive/MyDrive/UWaterloo/4A/MSE 446/Project/best_yolo_model.pt


In [6]:
# Set up paths
from pathlib import Path

# Update this path to point to your cleaned dataset
data_yaml_path = Path("/content/drive/MyDrive/UWaterloo/4A/MSE 446/Project/kagglehub/datasets/pkdarabi/bone-fracture-detection-computer-vision-project/versions/2/BoneFractureYolo8/data.yaml")

# # Verify the data.yaml file exists and is correct
# with open(data_yaml_path, 'r') as f:
#     data_config = yaml.safe_load(f)
#     print("Dataset Configuration:")
#     print(f"  Number of classes: {data_config['nc']}")
#     print(f"  Class names: {data_config['names']}")
#     print(f"  Train path: {data_config['train']}")
#     print(f"  Val path: {data_config['val']}")
#     print(f"  Test path: {data_config['test']}")

In [1]:

import yaml
from pathlib import Path

# Full dataset folder from Kaggle cache
# dataset_dir = Path(r"/content/drive/MyDrive/UWaterloo/4A/MSE 446/Project/kagglehub/datasets/pkdarabi/bone-fracture-detection-computer-vision-project/versions/2/bone fracture detection.v4-v4.yolov8/data.yaml")
dataset_dir = Path(r"/content/drive/MyDrive/UWaterloo/4A/MSE 446/Project/kagglehub/datasets/pkdarabi/bone-fracture-detection-computer-vision-project/versions/2/bone fracture detection.v4-v4.yolov8/data.yaml")



# Path to the YOLO dataset config
data_yaml_path = dataset_dir / "data.yaml"

# Verify file exists
if not data_yaml_path.exists():
    raise FileNotFoundError(f"Dataset YAML not found: {data_yaml_path}")

# Load dataset configuration
with open(data_yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

print("Dataset Configuration:")
print(f"  Number of classes: {data_config['nc']}")
print(f"  Class names: {data_config['names']}")
print(f"  Train path: {data_config['train']}")
print(f"  Val path: {data_config['val']}")
print(f"  Test path: {data_config['test']}")

FileNotFoundError: Dataset YAML not found: /content/drive/MyDrive/UWaterloo/4A/MSE 446/Project/kagglehub/datasets/pkdarabi/bone-fracture-detection-computer-vision-project/versions/2/bone fracture detection.v4-v4.yolov8/data.yaml/data.yaml

In [10]:
from google.colab import drive
from ultralytics import YOLO
import os

# Mount Drive
drive.mount('/content/drive')

# Set path to saved model
best_model_path = "/content/drive/MyDrive/UWaterloo/4A/MSE 446/Project/best_yolo_model.pt"

# Load model
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    print(f"Loaded best model from: {best_model_path}")
else:
    print("Model not found. Check path.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded best model from: /content/drive/MyDrive/UWaterloo/4A/MSE 446/Project/best_yolo_model.pt


In [13]:
from pathlib import Path
from ultralytics import YOLO
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Path to saved best model in Drive
best_model_path = Path("/content/drive/MyDrive/UWaterloo/4A/MSE 446/Project/best_yolo_model.pt")

# Load model
if best_model_path.exists():
    model = YOLO(str(best_model_path))
    print(f"Loaded best model from: {best_model_path}")
else:
    raise FileNotFoundError(f"Model not found at {best_model_path}")

# Validate the model
metrics = model.val(data=data_yaml_path)

print("\nValidation Metrics:")
print(f"  mAP50: {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded best model from: /content/drive/MyDrive/UWaterloo/4A/MSE 446/Project/best_yolo_model.pt


NameError: name 'data_yaml_path' is not defined

In [27]:
# Validate the model on validation set
metrics = model.val(data=data_yaml_path)
print("\nValidation Metrics:")
print(f"  mAP50: {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")

Ultralytics 8.4.25 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,583,517 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1.9±1.2 MB/s, size: 12.8 KB)
val: Scanning /kaggle/input/bone-fracture-detection-computer-vision-project/bone fracture detection.v4-v4.yolov8/valid/labels... 348 images, 175 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 348/348 99.9it/s 3.5s
WARNING ⚠️ val: Cache directory /kaggle/input/bone-fracture-detection-computer-vision-project/bone fracture detection.v4-v4.yolov8/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 4.9it/s 4.5s
                   all        348        204      0.355      0.297      0.279      0.105
        elbow positive         28         29      0.154      0.138      0.111     0.0321
      fingers positive         41         48      0.262      0.208    

In [29]:
# Run inference on a few test images to visualize predictions
import glob
import cv2

# Get a few test images
test_images_dir = "BoneFractureYolo8/test/images"
test_images = glob.glob(os.path.join(test_images_dir, "*.jpg"))[:5]  # Get first 5 images

# Run predictions
for img_path in test_images:
    results = model.predict(img_path, save=True, conf=0.25)
    print(f"Processed: {img_path}")